<a href="https://colab.research.google.com/github/AndiSyafarIlham/online-retail-analysis/blob/main/notebooks/01_cleaning_online_retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# =====================================================
# CELL 1: SETUP & UPLOAD DATA
# Proyek: Analisis Online Retail (UCI, 2010-2011)
# =====================================================

# 1. Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import warnings

# 2. Pengaturan tampilan
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Library berhasil di-import")
print(f"Pandas : {pd.__version__}")
print(f"NumPy  : {np.__version__}")
print("-" * 50)

# 3. Upload file dataset
print("Silakan pilih file Online Retail (.xlsx atau .csv)...")
uploaded = files.upload()

# 4. Simpan nama file untuk dipakai di cell berikutnya
file_name = list(uploaded.keys())[0]
file_size_mb = len(uploaded[file_name]) / (1024 * 1024)

print("-" * 50)
print(f"File berhasil diunggah : {file_name}")
print(f"Ukuran file            : {file_size_mb:.2f} MB")

Library berhasil di-import
Pandas : 2.2.3
NumPy  : 2.4.6
--------------------------------------------------
Silakan pilih file Online Retail (.xlsx atau .csv)...


Saving Online Retail.xlsx to Online Retail.xlsx
--------------------------------------------------
File berhasil diunggah : Online Retail.xlsx
Ukuran file            : 22.62 MB


In [11]:
# =====================================================
# CELL 2: LOAD DATA MENTAH
# =====================================================

# 1. Baca file (xlsx butuh waktu sekitar 30-90 detik, tunggu sampai selesai)
if file_name.lower().endswith(".xlsx"):
    df_raw = pd.read_excel(file_name)
else:
    df_raw = pd.read_csv(file_name, encoding="ISO-8859-1")

# 2. Samakan nama kolom (jaga-jaga kalau versi dataset berbeda)
df_raw = df_raw.rename(columns={
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID"
})

# 3. Ringkasan awal
print(f"Jumlah baris  : {df_raw.shape[0]:,}")
print(f"Jumlah kolom  : {df_raw.shape[1]}")
print("-" * 50)
print("Tipe data per kolom:")
print(df_raw.dtypes)
print("-" * 50)
print("5 baris pertama:")
df_raw.head()

Jumlah baris  : 541,909
Jumlah kolom  : 8
--------------------------------------------------
Tipe data per kolom:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object
--------------------------------------------------
5 baris pertama:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


In [9]:
# =====================================================
# CELL 3: CEK KUALITAS DATA (SEBELUM CLEANING)
# =====================================================

print("NILAI KOSONG PER KOLOM")
print(df_raw.isna().sum())
print("-" * 50)

print("MASALAH LAIN")
print(f"Baris duplikat                : {df_raw.duplicated().sum():,}")
print(f"Quantity <= 0                 : {(df_raw['Quantity'] <= 0).sum():,}")
print(f"UnitPrice <= 0                : {(df_raw['UnitPrice'] <= 0).sum():,}")
print(f"Invoice pembatalan (awalan C) : {df_raw['InvoiceNo'].astype(str).str.startswith('C').sum():,}")
print("-" * 50)

print("CAKUPAN DATA")
print(f"Periode  : {df_raw['InvoiceDate'].min()} s/d {df_raw['InvoiceDate'].max()}")
print(f"Negara   : {df_raw['Country'].nunique()}")
print(f"Pelanggan unik (ber-ID) : {df_raw['CustomerID'].nunique():,}")
print(f"Produk unik (StockCode) : {df_raw['StockCode'].nunique():,}")

NILAI KOSONG PER KOLOM
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
--------------------------------------------------
MASALAH LAIN
Baris duplikat                : 5,268
Quantity <= 0                 : 10,624
UnitPrice <= 0                : 2,517
Invoice pembatalan (awalan C) : 9,288
--------------------------------------------------
CAKUPAN DATA
Periode  : 2010-12-01 08:26:00 s/d 2011-12-09 12:50:00
Negara   : 38
Pelanggan unik (ber-ID) : 4,372
Produk unik (StockCode) : 4,070


In [10]:
# =====================================================
# CELL 4: DATA CLEANING + LOG JUMLAH BARIS
# =====================================================

log = []
def catat(langkah, data):
    log.append({"langkah": langkah, "jumlah_baris": len(data)})

df = df_raw.copy()
catat("Data awal", df)

# 1. Perbaiki tipe data
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# 2. Pisahkan invoice pembatalan (diawali huruf C) ke tabel sendiri
mask_cancel = df["InvoiceNo"].str.startswith("C")
df_cancel = df[mask_cancel].copy()
df = df[~mask_cancel].copy()
catat("Pisahkan pembatalan", df)

# 3. Hapus baris duplikat
df = df.drop_duplicates()
catat("Hapus duplikat", df)

# 4. Hapus Quantity atau UnitPrice yang nol/negatif
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
catat("Hapus Quantity/UnitPrice <= 0", df)

# 5. Rapikan Description, buang yang kosong
df["Description"] = df["Description"].astype(str).str.strip().str.upper()
df = df[~df["Description"].isin(["NAN", ""])]
catat("Hapus Description kosong", df)

# 6. Buang kode non-produk (ongkir, biaya admin, dll)
kode_non_produk = ["POST", "D", "M", "DOT", "CRUK", "BANK CHARGES", "AMAZONFEE", "S", "B", "PADS"]
mask_non = df["StockCode"].astype(str).isin(kode_non_produk)
print("Kode non-produk yang dibuang per jenis:")
print(df.loc[mask_non, "StockCode"].value_counts())
df = df[~mask_non]
catat("Hapus kode non-produk", df)

# 7. Tabel log
df_log = pd.DataFrame(log)
df_log["baris_dibuang"] = df_log["jumlah_baris"].diff().abs().fillna(0).astype(int)
print("-" * 50)
df_log

Kode non-produk yang dibuang per jenis:
StockCode
POST            1126
DOT              706
M                316
BANK CHARGES      12
PADS               3
AMAZONFEE          2
S                  2
B                  1
Name: count, dtype: int64
--------------------------------------------------


,langkah,jumlah_baris,baris_dibuang
0,Data awal,541909,0
1,Pisahkan pembatalan,532621,9288
2,Hapus duplikat,527390,5231
3,Hapus Quantity/UnitPrice <= 0,524878,2512
4,Hapus Description kosong,524878,0
5,Hapus kode non-produk,522710,2168


In [12]:
# =====================================================
# CELL 5: FEATURE ENGINEERING
# =====================================================

df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["Bulan"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["Hari"] = df["InvoiceDate"].dt.day_name()
df["Jam"] = df["InvoiceDate"].dt.hour

# CustomerID kosong tidak dibuang: dipakai untuk analisis penjualan,
# tapi ditandai supaya RFM nanti hanya memakai yang punya ID
df["CustomerID"] = df["CustomerID"].astype("Int64")
df["punya_customer_id"] = df["CustomerID"].notna()

print("RINGKASAN DATA BERSIH")
print(f"Jumlah baris          : {len(df):,}")
print(f"Total revenue         : {df['Revenue'].sum():,.2f}")
print(f"Invoice unik          : {df['InvoiceNo'].nunique():,}")
print(f"Pelanggan unik        : {df['CustomerID'].nunique():,}")
print(f"Baris tanpa CustomerID: {(~df['punya_customer_id']).sum():,}")
print(f"Jumlah baris retur    : {len(df_cancel):,}")
df.head()

RINGKASAN DATA BERSIH
Jumlah baris          : 522,710
Total revenue         : 10,254,958.68
Invoice unik          : 19,776
Pelanggan unik        : 4,334
Baris tanpa CustomerID: 131,427
Jumlah baris retur    : 9,288


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Bulan,Hari,Jam,punya_customer_id
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12,Wednesday,8,True
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,Wednesday,8,True
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12,Wednesday,8,True
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,Wednesday,8,True
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,Wednesday,8,True


In [13]:
# =====================================================
# CELL 6: SIMPAN HASIL CLEANING
# =====================================================
import os

# 1. Data bersih (untuk dipakai sendiri: SQL & dashboard)
df.to_csv("online_retail_clean.csv", index=False)

# 2. Data bersih versi terkompresi (untuk GitHub, di bawah 25 MB)
df.to_csv("online_retail_clean.csv.gz", index=False, compression="gzip")

# 3. Data pembatalan/retur
df_cancel.to_csv("online_retail_cancel.csv", index=False)

# 4. Log cleaning (bahan README)
df_log.to_csv("cleaning_log.csv", index=False)

# 5. Cek ukuran file
for f in ["online_retail_clean.csv", "online_retail_clean.csv.gz",
          "online_retail_cancel.csv", "cleaning_log.csv"]:
    print(f"{f:32s} {os.path.getsize(f) / (1024*1024):6.2f} MB")

# 6. Unduh semuanya ke komputer
for f in ["online_retail_clean.csv", "online_retail_clean.csv.gz",
          "online_retail_cancel.csv", "cleaning_log.csv"]:
    files.download(f)

online_retail_clean.csv           59.10 MB
online_retail_clean.csv.gz         8.60 MB
online_retail_cancel.csv           0.80 MB
cleaning_log.csv                   0.00 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>